In [1]:
import pandas as pd
import os
import io
import re
import paramiko
from io import BytesIO
from ftplib import FTP
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# def ConnectFTP(server, username, password):
#     """
#     Descripcion: Esta funcion establece una conexion FTP con el servidor especificado.
#     Parametros:
#     server (str): La direccion del servidor FTP.
#     username (str): El nombre de usuario para la conexion FTP.
#     password (str): La contrasena para la conexion FTP.
#     Retorna: Un objeto FTP conectado al servidor.
#     """
#     ftp = FTP(timeout=60)                
#     ftp.connect(server, 21)               
#     ftp.login(user=username, passwd=password)

#     ftp.set_pasv(True)                    
#     ftp.voidcmd("TYPE I")                 

#     print(f"Conectado a {server}")
#     return ftp

# def UploadCsvToFTP(df, path):
#     """
#     Descripcion: Esta funcion sube un DataFrame de pandas como un archivo CSV a un servidor FTP.
#     Parametros:
#     df (DataFrame): El DataFrame que se desea subir.
#     path (str): La ruta en el servidor FTP donde se guardará el archivo CSV.
#     Retorna: None
#     """
#     csv_buffer = io.BytesIO()
#     df.to_csv(csv_buffer, index=False, encoding='utf-8')
#     csv_buffer.seek(0)
#     ftp = ConnectFTP(os.getenv('ftp_server'),os.getenv('ftp_user'),os.getenv('ftp_password'))
#     # remote_path = f"/pruebas/csv/{path}"
#     ftp.storbinary(f"STOR {path}", csv_buffer)
#     ftp.quit()
#     print("Archivo subido correctamente:", path)

# def ReadCsvFromFTP(aux, namearc ):
#     '''
#     Descripcion: Esta funcion lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
#     Parametros:
#     ftp (FTP): Un objeto FTP conectado al servidor.
#     remote_file_path (str): La ruta del archivo CSV en el servidor FTP.
#     Retorna: Un DataFrame de pandas que contiene los datos del archivo CSV.
#     '''
#     ftp = ConnectFTP(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
#     with BytesIO() as bio:
#         ftp.retrbinary(f'RETR /pruebas/csv/{aux}/{namearc}', bio.write)
#         bio.seek(0)
#         df = pd.read_csv(bio)
#     return df

def ConnectSFTP(server, username, password, port=22):
    """
    Establece una conexión SFTP mediante SSH.

    Retorna:
        ssh: cliente SSH necesario para cerrar correctamente la conexión.
        sftp: cliente utilizado para navegar y transferir archivos.
    """
    ssh = paramiko.SSHClient()
    ssh.load_system_host_keys()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    ssh.connect(
        hostname=server,
        port=port,
        username=username,
        password=password,
        timeout=60,
    )

    sftp = ssh.open_sftp()

    print(f"Conectado mediante SFTP a {server}:{port}")
    return ssh, sftp

def UploadCsvToSFTP(df, path):
    """
    Convierte un DataFrame a CSV en memoria y lo sube mediante SFTP.
    """
    csv_buffer = io.BytesIO()
    df.to_csv(
        csv_buffer,
        index=False,
        encoding="utf-8"
    )
    csv_buffer.seek(0)
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    # remote_path = f"riecs/pruebas2/csv/{path}"
    try:
        sftp.putfo(csv_buffer, path)
        print("Archivo subido correctamente:", path)
    finally:
        sftp.close()
        ssh.close()

def ReadCsvFromSFTP(remote_file_path):
    """
    Lee un archivo CSV desde un servidor SFTP y lo carga
    en un DataFrame de pandas.

    Parámetros:
        remote_file_path (str):
            Ruta completa del archivo CSV dentro del servidor.

    Retorna:
        pandas.DataFrame:
            DataFrame con el contenido del archivo CSV.
    """
    ssh, sftp = ConnectSFTP(
        os.getenv("ftp_server"),
        os.getenv("ftp_user"),
        os.getenv("ftp_password")
    )
    try:
        with sftp.open(remote_file_path, mode="rb") as remote_file:
            df = pd.read_csv(remote_file)
        return df
    finally:
        sftp.close()
        ssh.close()

In [3]:
def Mod_Sociodemograficos(df):
    list_columns = ['ComunOcupacionId', 'SexoId', 'ComunEstadoCivilId', 'ComunEscolaridadId', 'ComunEstratoSocialId']
    df[list_columns] = df[list_columns].fillna('Sin Dato')
    df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
                                                    'Sin ocupación': 2, 'Sin ocupacion': 2,
                                                    'Con actividad laboral': 3,
                                                    'Estudiante': 4,
                                                    'Hogar': 5,
                                                    'Sin Dato': 0}).astype('Int8')
    
    df['SexoId'] = df['SexoId'].replace({'Hombre': 1,
                                        'Mujer': 2,
                                        'Sin Dato': 0}).astype('Int8')
    
    df['ComunEstadoCivilId'] = df['ComunEstadoCivilId'].replace({'Soltero(a)': 1,
                                                                'Casado(a)': 2,
                                                                'Divorciado(a)': 3,
                                                                'Unión libre': 4,
                                                                'Unión Libre': 4,
                                                                'Separado(a)': 5,
                                                                'Viudo(a)': 6,
                                                                'Sin Dato': 0}).astype('Int8')
    df['ComunEscolaridadId'] = df['ComunEscolaridadId'].replace({'Sin Estudios': 1,
                                                                'Primaria': 2,
                                                                'Secundaria': 3,
                                                                'Preparatoria o Carrera Técnica': 4,
                                                                'Estudios Superiores': 5,
                                                                'Estudios de Posgrado': 6,
                                                                'Sin Dato': 0}).astype('Int8')
    
    df['ComunEstratoSocialId'] = df['ComunEstratoSocialId'].replace({'Muy Bajo': 1,
                                                                    'Bajo': 2,
                                                                    'Medio Bajo': 3,
                                                                    'Medio Alto': 4,
                                                                    'Alto': 5,
                                                                    'Sin Dato': 0}).astype('Int8')
    
    df['CentroCostoId'] = df['CentroCostoId'].fillna(99999).astype('Int32')
    return df

def DataClean(df):
    columnas_a_convertir3 = [
        "SexoId",
        "Migracion",
        "ComunEstadoCivilId",
        "ComunEscolaridadId",
        "ComunOcupacionId",
        "ComunEstratoSocialId",
        "PerteneceComunidadLGBTTTI",
        "PerteneceComunidadIndigena",
        "PoblacionAfromexicanaAfroamericana",
        "DiscapacidadPerceptual",
        "Edad_años",
        "ConsumoDeDrogas",
        "ConsumoDeBebidasAlcoholicas",
        "ConsumoDeTabaco",
        "Ludopatia",
        "Otro",
        "TrastornosMentales",
        "Depresion",
        "Psicosis",
        "Epilepsia",
        "Demencia",
        "Autolesion",
        "Suicidio",
        "Ansiedad",
    ]
    for columna in columnas_a_convertir3:
        if columna in df.columns:
            df[columna] = pd.to_numeric(df[columna], errors="coerce").astype("Int8")

    df['Año'] = pd.to_numeric(df['Año'], errors="coerce").astype("Int16")
    df['FolioId'] = pd.to_numeric(df['FolioId'], errors="coerce").astype("Int32")

    df = df.drop('Mes', axis=1)
    df = df[df['Estado'].notna() & (df['Estado'] != "")]
    # df.drop(df[df['Año'] < 2014].index, inplace=True)
    
    return df

def limpieza_datasets (df):
        #Se definen las columnas que serán filtradas del dataset
    filtro = [
        "ConsumoDeDrogas",
        "ConsumoDeBebidasAlcoholicas",
        "ConsumoDeTabaco",
        "Ludopatia",
        "Otro",
        "TrastornosMentales",
        "Depresion",
        "Psicosis",
        "Epilepsia",
        "Demencia",
        "Autolesion",
        "Suicidio",
        "Ansiedad",
    ]
    # Verificar que las columnas existan
    columnas_existentes = [col for col in filtro if col in df.columns]
        # Filtrar filas en el mismo DataFrame (reasignando al índice filtrado)
    df.drop(df[~df[columnas_existentes].ge(1).any(axis=1)].index, inplace=True)
    return df

In [4]:
def main():
    try: 
        file = 'Motivos_Consulta.csv'
        df = ReadCsvFromSFTP(f'riecs/pruebas2/csv/carter/{file}')
    except Exception as e:
        print("Error en Optimizacion_MotivoConsulta", e)
        return 
    try:
        df = Mod_Sociodemograficos(df)
    except Exception as e:
        print("Error en Optimizacion_MotivoConsulta", e)
        return
    try:
        df = DataClean(df)
    except Exception as e:
        print("Error en Optimizacion_MotivoConsulta", e)
        return
    try:
        df = limpieza_datasets(df)
        UploadCsvToSFTP(df, f'riecs/pruebas2/csv/carter/{file}')
    except Exception as e:
        print("Error en Optimizacion_MotivoConsulta", e)
        return

if __name__ == "__main__":
    main()

Conectado mediante SFTP a 192.168.62.150:22


C:\Users\franc\AppData\Local\Temp\ipykernel_5788\887815305.py:121: DtypeWarning: Columns (0,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(remote_file)
C:\Users\franc\AppData\Local\Temp\ipykernel_5788\3585569847.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['ComunOcupacionId'] = df['ComunOcupacionId'].replace({'Pensionado o jubilado': 1,
C:\Users\franc\AppData\Local\Temp\ipykernel_5788\3585569847.py:11: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['SexoId'] = df['SexoI

Conectado mediante SFTP a 192.168.62.150:22
Archivo subido correctamente: riecs/pruebas2/csv/carter/Motivos_Consulta.csv
